# MLOps 01 — The big picture: from a notebook model to a monitored service

**Open `explainer.html` first.** This notebook walks the whole MLOps lifecycle **once**, at beginner depth.
Later topics go deep into one module each; every concept you meet here comes back there (that's the plan).

**The scenario.** *DeliverEase* (food delivery) shows customers "arrives in 34 min". A model predicts that number.
Your job is not just to train it, but to keep it trustworthy every day: when the monsoon arrives, the old model
starts being 10+ minutes wrong. You'll see it happen, detect it, retrain, and roll out the fix safely.

**How to work.** Read → run → tweak. Each `# TODO` is one short line, and every cell ends with `assert` checks.
The real implementation of each idea lives in `../../codebase/deliverease/`. You write the key line here,
then see the production version there.

| Cell | Module | You write |
|------|--------|-----------|
| 1 | M1 Foundations | look at the data first |
| 2–3 | M3 Experimentation | a baseline; the shared feature function |
| 4–5 | M3 Tracking & registry | read the best run; quality gate + promote |
| 6–7 | M5 / M4 Serving | call the API; batch scoring |
| 8–9 | M6 Monitoring | PSI drift by hand; live MAE |
| 10–11 | M7 Continual learning | champion vs challenger; canary share |
| 12 | M8 Trends | (reading + your own words) |

---
## Cell 0 — Setup (nothing to write)
Makes the codebase importable and creates a scratch folder `nb_workdir/` next to this notebook (git-ignored).

In [ ]:
import sys, shutil
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

CODEBASE = Path("../../codebase").resolve()
sys.path.insert(0, str(CODEBASE))
from deliverease.data import make_orders, FEATURES, TARGET
from deliverease.features import build_features, MODEL_INPUTS
from deliverease.tracking import Tracker
from deliverease.registry import Registry
from deliverease.train import train
from deliverease.pipeline import quality_gate, run_pipeline
from deliverease.monitor import psi, drift_report, should_retrain
from deliverease.rollout import Router, bucket

WORK = Path("nb_workdir")
shutil.rmtree(WORK, ignore_errors=True)        # start fresh each time you run the notebook
WORK.mkdir()
print("codebase:", CODEBASE)

---
## Cell 1 — Look at the data first · *M1, explainer §1*
Rule 1 of ML work: **look at real rows before writing any model code.** Each row is one past order.

In [ ]:
df = make_orders(2000, seed=0)                  # dry-season orders
display(df.head())

avg = None   # TODO: the average delivery time  ->  df[TARGET].mean()

print(f"average delivery: {avg:.1f} min, rainy orders: {df.is_raining.mean():.0%}")
assert 25 < avg < 45, "the mean of the delivery_min column"
print("✅ you looked at the data")

---
## Cell 2 — A baseline before any model · *M3, explainer §3*
The simplest possible "model": always predict the average from the training data. Any real model must beat it clearly.
**MAE** (mean absolute error) = on average, how many minutes off we are.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

baseline_pred = None   # TODO: np.full(len(test_df), train_df[TARGET].mean())
baseline_mae  = None   # TODO: mean_absolute_error(test_df[TARGET], baseline_pred)

print(f"baseline MAE: {baseline_mae:.2f} minutes")
assert 7 < baseline_mae < 12
print("✅ now we know what 'good' has to beat")

---
## Cell 3 — One feature function for training AND serving · *M3, explainer §3*
`build_features` turns raw order columns into model inputs (e.g. hour → `is_peak`). Training and the API both import
this same function, which prevents **training-serving skew**.

In [ ]:
X = None   # TODO: build_features(df)

display(X.head(3))
assert list(X.columns) == MODEL_INPUTS
print("✅ same function will run inside the API — open codebase/deliverease/features.py")

---
## Cell 4 — Train and track the experiment · *M3, explainer §4*
`train()` fits a model, compares it with the baseline and **logs the run** (params, data version, metrics, file).
Run it twice with different settings, then ask the tracker which run was best.

In [ ]:
tracker = Tracker(WORK / "runs")
r1 = train(df, WORK / "artifacts_a", tracker, params={"max_depth": 2})
r2 = train(df, WORK / "artifacts_b", tracker, params={"max_depth": 4})

best = None   # TODO: tracker.best("mae")      (lower MAE is better)

print(pd.DataFrame([{**r["params"], **r["metrics"]} for r in tracker.runs()])[["max_depth", "data_version", "mae", "baseline_mae"]])
assert best["metrics"]["mae"] == min(r1["metrics"]["mae"], r2["metrics"]["mae"])
print("✅ best run:", best["run_id"], best["metrics"])

---
## Cell 5 — Registry + quality gate · *M3/M4, explainer §4*
Register the best model as a new **version**. Promote it to **Production** only if it passes the gate
(MAE under 6 min and ≥ 30% better than the baseline). There's no production model yet, so `champion_mae=None`.

In [ ]:
registry = Registry(WORK / "registry")
version = registry.register(best["artifact"], best["run_id"], best["metrics"])

ok, reason = None, None   # TODO: quality_gate(best["metrics"], champion_mae=None)
if ok:
    pass   # TODO: registry.promote(version, "Production")

print(f"v{version}: gate {reason}")
assert registry.get("Production")["version"] == version
print("✅ v1 is live. Serving code will ask the registry, never a laptop file")

---
## Cell 6 — Online serving: the model behind an API · *M5, explainer §5*
`create_app` builds the FastAPI service. `TestClient` lets us call it without starting a server.
Every response carries the model version, so every prediction is traceable.

In [ ]:
from fastapi.testclient import TestClient
from deliverease.serve import create_app
client = TestClient(create_app(str(WORK / "registry")))

order = None   # TODO: a dict with distance_km=3.5, hour=20, items=2, prep_min=12, is_raining=0
response = client.post("/predict", json=order)

print(response.json())
assert response.status_code == 200 and response.json()["model_version"] == version
bad = client.post("/predict", json={**order, "distance_km": -1})
assert bad.status_code == 422, "invalid input must be rejected before reaching the model"
print("✅ live API answered; invalid input rejected with 422")

---
## Cell 7 — Batch (offline) prediction · *M4/M5, explainer §5*
Tomorrow's scheduled orders don't need an API: score the whole file overnight.

In [ ]:
from deliverease.batch import score_file
make_orders(300, seed=7)[FEATURES].to_csv(WORK / "tomorrow.csv", index=False)
n_scored = score_file(WORK / "tomorrow.csv", WORK / "tomorrow_scored.csv", WORK / "registry")
scored = pd.read_csv(WORK / "tomorrow_scored.csv")

late = None   # TODO: how many orders have eta_min above 45?  ->  int((scored["eta_min"] > 45).sum())

print(f"scored {n_scored} orders, {late} expected to take > 45 min")
assert n_scored == 300 and isinstance(late, int)
print("✅ batch job done, no server needed")

---
## Cell 8 — The monsoon arrives: data drift · *M6, explainer §6*
**PSI** (Population Stability Index) compares a feature's distribution now vs. at training time.
`PSI = Σ (a% − e%) × ln(a% / e%)` over bins. Rule of thumb: < 0.1 ok · 0.1–0.25 watch · > 0.25 alert.
Compute it by hand for `prep_min`, then let the codebase do every feature.

In [ ]:
monsoon = make_orders(2000, seed=9, monsoon=True)

edges = np.quantile(df["prep_min"], np.linspace(0, 1, 11)); edges[0], edges[-1] = -np.inf, np.inf
e = np.histogram(df["prep_min"], edges)[0] / len(df)            # expected % per bin (training)
a = np.histogram(monsoon["prep_min"], edges)[0] / len(monsoon)  # actual % per bin (live)
e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)

my_psi = None   # TODO: np.sum((a - e) * np.log(a / e))

print(f"PSI prep_min by hand: {my_psi:.3f}   codebase: {psi(df['prep_min'], monsoon['prep_min']):.3f}")
assert abs(my_psi - psi(df["prep_min"], monsoon["prep_min"])) < 1e-6
report = drift_report(df, monsoon, FEATURES)
display(report)
print("✅ drift detected without a single label:", should_retrain(report))

---
## Cell 9 — Labels arrive: is the model actually worse? · *M6, explainer §6*
Drift says *the inputs changed*. Only real delivery times (labels, a few hours later) say *the model is wrong*.

In [ ]:
import joblib
prod_model = joblib.load(registry.get("Production")["path"])

live_mae = None   # TODO: mean_absolute_error(monsoon[TARGET], prod_model.predict(build_features(monsoon)))

retrain, why = should_retrain(report, live_mae=live_mae)
print(f"live MAE in the monsoon: {live_mae:.1f} min (was {best['metrics']['mae']:.1f})  ->  retrain? {retrain}: {why}")
assert live_mae > 3 * best["metrics"]["mae"]
print("✅ concept drift: same kind of inputs, but deliveries got slower in ways no feature explains")

---
## Cell 10 — Retrain: champion vs challenger · *M4/M7, explainer §7*
The pipeline retrains on monsoon data and scores **both** models on the **same fresh monsoon data**.
The new one (challenger) only replaces production (champion) if it wins.

In [ ]:
result = run_pipeline(WORK, seed=3, monsoon=True)

champion_mae, challenger_mae = None, None   # TODO: read both from result  ->  result["champion_mae"], result["challenger_mae"]

print(f"champion (old) {champion_mae:.2f}  vs  challenger (new) {challenger_mae:.2f}  ->  promoted: {result['promoted']} ({result['reason']})")
assert challenger_mae < champion_mae and result["promoted"]
display(pd.DataFrame(registry.versions())[["version", "stage", "run_id"]])
print("✅ old version archived, new version live")

---
## Cell 11 — Roll out safely: canary · *M7, explainer §7*
Instead of switching everyone at once, send 10% of orders to the new model first. Routing by a hash of the order id
means the same customer always gets the same model.

In [ ]:
old = joblib.load(registry.versions()[0]["path"])
new = joblib.load(registry.get("Production")["path"])

router = Router(current=old, candidate=new, canary_percent=None)   # TODO: replace None with 10

X1 = build_features(monsoon.head(1))
served = [router.predict(f"order-{i}", X1)[1] for i in range(2000)]
share = served.count("candidate") / len(served)
print(f"{share:.1%} of orders served by the new model")
assert 0.07 < share < 0.13
print("✅ canary at ~10%. Next step in real life: watch its metrics, then 25% → 50% → 100%")

---
## Cell 12 — Trends + say it in your own words · *M8, explainer §8*
M8 (trends) is reading, not code: model compression, ML in browsers and phones, edge, federated learning.
Write 1–2 sentences each in the cell below, then check against the explainer:
1. Why is ML in production harder than normal software? Name the three things that can change.
2. What is training-serving skew, and how did this codebase prevent it?
3. Data drift vs concept drift: which one needs labels to detect?
4. Why must champion and challenger be scored on the *same* data? (This codebase first had that bug.)
5. Shadow vs canary: which one carries zero user risk?

*Your answers:*

1.
2.
3.
4.
5.